In [1]:
import pandas as pd
import geopandas as gpd

# Load labeled warnings
ffw = gpd.read_parquet("../data/ffw_labeled_ws2024.parquet")

print(ffw.shape)
print(ffw[["WARNING_ID", "ISSUED", "EXPIRED", "WFO", "verified", 
           "false_alarm_fraction", "AREA_KM2", "centroid_lon", "centroid_lat"]].head(10))

# Quick summary
print("\nVerification breakdown:")
print(ffw["verified"].value_counts())

# Helene period
helene = ffw[(ffw["ISSUED"] >= "2024-09-26") & (ffw["ISSUED"] <= "2024-09-30")]
print(f"\nHelene warnings: {len(helene)}")
print(helene[["WARNING_ID", "ISSUED", "WFO", "verified", "false_alarm_fraction"]].head(10))

(3700, 38)
   WARNING_ID                    ISSUED                   EXPIRED  WFO  \
0           0 2024-04-01 04:13:00+00:00 2024-04-01 07:15:00+00:00  PSR   
1           1 2024-04-09 17:59:00+00:00 2024-04-09 20:30:00+00:00  FWD   
2           2 2024-04-23 20:32:00+00:00 2024-04-23 23:45:00+00:00  MAF   
3           3 2024-04-28 02:21:00+00:00 2024-04-28 05:30:00+00:00  SJT   
4           4 2024-04-28 09:47:00+00:00 2024-04-28 12:45:00+00:00  FWD   
5           5 2024-04-30 23:06:00+00:00 2024-05-01 02:15:00+00:00  OUN   
6           6 2024-04-16 01:35:00+00:00 2024-04-16 04:37:00+00:00  LMK   
7           7 2024-04-25 22:44:00+00:00 2024-04-26 02:17:00+00:00  EAX   
8           8 2024-04-11 21:58:00+00:00 2024-04-12 02:00:00+00:00  PBZ   
9           9 2024-04-08 17:53:00+00:00 2024-04-08 18:45:00+00:00  JAN   

   verified  false_alarm_fraction     AREA_KM2  centroid_lon  centroid_lat  
0     False              1.000000   330.625738   -113.551041     33.734302  
1      True         

In [2]:
# Best verified candidate: high n_lsr_inside, reasonable area, not too large
verified_candidates = ffw[ffw["verified"] == True].sort_values("n_lsr_inside", ascending=False)
print("Top verified candidates:")
print(verified_candidates[["WARNING_ID","ISSUED","WFO","n_lsr_inside",
                            "first_lsr_minutes","AREA_KM2","centroid_lon","centroid_lat"]].head(5))

# Best unverified candidate: clean false alarm, reasonable area
unverified_candidates = ffw[ffw["verified"] == False].sort_values("AREA_KM2")
print("\nTop unverified candidates (small-medium area):")
print(unverified_candidates[["WARNING_ID","ISSUED","WFO","AREA_KM2",
                              "centroid_lon","centroid_lat"]].head(5))

# Helene verified
helene_verified = helene[helene["verified"] == True].sort_values("n_lsr_inside", ascending=False)
print("\nHelene verified candidates:")
print(helene_verified[["WARNING_ID","ISSUED","WFO","n_lsr_inside",
                        "AREA_KM2","centroid_lon","centroid_lat"]].head(5))

Top verified candidates:
      WARNING_ID                    ISSUED  WFO  n_lsr_inside  \
40            40 2024-04-11 21:32:00+00:00  PBZ            48   
190          190 2024-04-11 05:09:00+00:00  TAE            47   
200          200 2024-04-02 19:52:00+00:00  PBZ            46   
2332        2332 2024-07-01 15:41:00+00:00  EAX            42   
192          192 2024-04-11 07:04:00+00:00  TAE            37   

      first_lsr_minutes      AREA_KM2  centroid_lon  centroid_lat  
40                 27.0   5134.686722    -80.231875     40.592355  
190               114.0  12316.549613    -84.569943     30.211173  
200               128.0  15376.125496    -81.262107     40.118541  
2332               34.0   4735.957534    -94.572592     38.895540  
192                 9.0   3692.610759    -84.158484     30.470694  

Top unverified candidates (small-medium area):
      WARNING_ID                    ISSUED  WFO  AREA_KM2  centroid_lon  \
2447        2447 2024-08-07 16:18:00+00:00  CHS  2.60

In [3]:
import pandas as pd
import geopandas as gpd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from herbie import Herbie
import warnings
warnings.filterwarnings("ignore")

# Load labeled warnings
ffw = gpd.read_parquet("../data/ffw_labeled_ws2024.parquet")

# Select 3 case study warnings
CASE_IDS = {
    "verified":    40,
    "unverified":  2521,
    "helene":      3389,
}

cases = {name: ffw[ffw["WARNING_ID"] == wid].iloc[0] 
         for name, wid in CASE_IDS.items()}

# Print summary of each
for name, row in cases.items():
    print(f"\n{'='*50}")
    print(f"  {name.upper()} — WARNING_ID {row['WARNING_ID']}")
    print(f"  WFO:      {row['WFO']}")
    print(f"  Issued:   {row['ISSUED']}")
    print(f"  Expired:  {row['EXPIRED']}")
    print(f"  Area:     {row['AREA_KM2']:.1f} km²")
    print(f"  Verified: {row['verified']}  |  FA fraction: {row['false_alarm_fraction']:.3f}")
    print(f"  Centroid: ({row['centroid_lat']:.3f}°N, {row['centroid_lon']:.3f}°E)")


  VERIFIED — WARNING_ID 40
  WFO:      PBZ
  Issued:   2024-04-11 21:32:00+00:00
  Expired:  2024-04-12 01:30:00+00:00
  Area:     5134.7 km²
  Verified: True  |  FA fraction: 0.702
  Centroid: (40.592°N, -80.232°E)

  UNVERIFIED — WARNING_ID 2521
  WFO:      BOU
  Issued:   2024-08-08 23:07:00+00:00
  Expired:  2024-08-09 01:18:00+00:00
  Area:     4.8 km²
  Verified: False  |  FA fraction: 1.000
  Centroid: (39.541°N, -105.161°E)

  HELENE — WARNING_ID 3389
  WFO:      CAE
  Issued:   2024-09-26 18:02:00+00:00
  Expired:  2024-09-26 20:00:00+00:00
  Area:     2386.6 km²
  Verified: True  |  FA fraction: 0.855
  Centroid: (33.975°N, -81.005°E)
